# Two-Stage Cancer Classification Pipeline

This notebook implements a two-stage machine learning pipeline to classify cancer samples into five categories:
- **Healthy**
- **Screening stage cancer**
- **Early stage cancer**
- **Mid stage cancer**
- **Late stage cancer**

## Why a Two-Stage Approach?

A direct 5-class classifier struggles with the severe class imbalance in this dataset — healthy samples are rare compared to cancer-stage samples. A single model tends to ignore minority classes entirely.

To address this, we split the problem into two sequential steps:
1. **Stage 1 Classifier** — Distinguishes between `healthy+screening` (combined), `early stage`, `mid stage`, and `late stage` cancer. This coarser grouping balances class counts and handles the bulk of the classification work.
2. **Stage 2 Classifier** — Operates only on samples predicted as `healthy+screening` by Stage 1, then separates them into `healthy` vs `screening stage cancer`. This second step targets the most clinically critical distinction.

Two algorithms are benchmarked at each stage to compare performance under class imbalance conditions.

# 1. Data Loading and Exploratory Analysis

We load the training set and perform a quick structural inspection. The dataset contains approximately 350 genomic/biological features per sample. Understanding the class distribution early is essential, as it directly informs our preprocessing and modelling decisions.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

cancer = pd.read_csv("Train_Set.csv")

print(f"Dataset shape: {cancer.shape}")
cancer.info()
cancer.describe()

### Class Distribution

We visualise the frequency of each class label to quantify the imbalance. As expected in a real-world clinical dataset, healthy samples are underrepresented — this confirms the need for imbalance-aware modelling strategies such as class weighting and stratified cross-validation.

In [ ]:
class_counts = cancer["class_label"].value_counts()

plt.figure(figsize=(8, 6))
class_counts.plot(kind='bar', color='skyblue')
plt.title('Class Frequency in Training Data')
plt.xlabel('Class Label')
plt.ylabel('Count')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

# 2. Data Preprocessing

## 2.1 Training Data Preprocessing

We prepare two separate feature matrices — one for each classification stage.

**Stage 1 preprocessing:**
- `healthy` and `screening stage cancer` are merged into a single class (`0`) to form a 4-class problem
- Features are L2-normalised row-wise using `sklearn.preprocessing.normalize` to account for varying feature magnitudes
- An 80/20 train-validation split is applied with `random_state=100` for reproducibility

**Stage 2 preprocessing:**
- Only samples originally labelled `healthy` or `screening stage cancer` are retained
- Features are standardised using `StandardScaler` (fit on training data only) to satisfy the assumptions of distance-sensitive models like XGBoost and Naive Bayes

In [ ]:
from sklearn import preprocessing
from sklearn.model_selection import GridSearchCV, train_test_split, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay, make_scorer, recall_score
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier

# ── Stage 1: Merge healthy and screening into one class ──
combine_class = cancer.copy()
cancer_class = {
    "healthy": 0,
    "screening stage cancer": 0,
    "early stage cancer": 2,
    "mid stage cancer": 3,
    "late stage cancer": 1
}
combine_class['class_label'] = combine_class['class_label'].map(cancer_class)
new_class = ['healthy+screening', 'late stage cancer', 'early stage cancer', 'mid stage cancer']

X = combine_class.iloc[:, :-1].values
y = combine_class.iloc[:, -1].values

# L2 normalisation
X = preprocessing.normalize(X, axis=1)

# Train-validation split
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.20, random_state=100)

# ── Stage 2: Isolate healthy vs screening samples ──
cancer_2stage = cancer.copy()
filtered_train = cancer_2stage[cancer_2stage['class_label'].isin(['healthy', 'screening stage cancer'])]

label_mapping = {"healthy": 0, "screening stage cancer": 1}
filtered_train = filtered_train.copy()
filtered_train['class_label'] = filtered_train['class_label'].map(label_mapping)

X_2 = filtered_train.drop(columns=['class_label']).values
y_2 = filtered_train['class_label'].values

# StandardScaler fit on training data only
scaler = StandardScaler()
X_train2 = scaler.fit_transform(X_2)

print("Stage 1 training set shape:", X_train.shape)
print("Stage 2 training set shape:", X_train2.shape)

## 2.2 Test Data Preprocessing

The same transformations applied to training data are mirrored on the test set. Critically, the `StandardScaler` fitted on the training data is reused here — applying `transform` rather than `fit_transform` — to prevent data leakage.

In [ ]:
# Load test set
test_data = pd.read_csv("Test_Set.csv")

# ── Stage 1 test preprocessing ──
combine_test = test_data.copy()
combine_test['class_label'] = combine_test['class_label'].map(cancer_class)

X_test = combine_test.iloc[:, :-1].values
y_test_combined = combine_test.iloc[:, -1].values
X_test = preprocessing.normalize(X_test, axis=1)

# ── Stage 2 test preprocessing ──
filtered_test = test_data[test_data['class_label'].isin(['healthy', 'screening stage cancer'])].copy()
filtered_test['class_label'] = filtered_test['class_label'].map(label_mapping)

X_test2 = filtered_test.drop(columns=['class_label']).values
y_test2 = filtered_test['class_label'].values

# Apply scaler fitted on training data — no leakage
X_test2 = scaler.transform(X_test2)

print("Stage 1 test set shape:", X_test.shape)
print("Stage 2 test set shape:", X_test2.shape)

# 3. Model Building, Training and Evaluation

## 3.1 Stage 1 Models: 4-Class Classification

Stage 1 classifies each sample into one of four groups: `healthy+screening`, `early stage cancer`, `mid stage cancer`, or `late stage cancer`. Two algorithms are evaluated:

- **Random Forest** — an ensemble method robust to high-dimensional data and capable of handling class imbalance via the `class_weight='balanced'` parameter
- **Neural Network** — a feedforward network with dropout regularisation, trained with class-weighted loss to address imbalance

Both models are evaluated on the held-out validation set and then on the unseen test set.

### 3.1.1 Random Forest

Hyperparameters are tuned using 5-fold cross-validation with `GridSearchCV`, optimising for weighted recall — a deliberate choice given class imbalance, where we care about correctly identifying all cancer stages, not just achieving overall accuracy.

In [ ]:
recall_scorer = make_scorer(recall_score, average="weighted")

rf_base = RandomForestClassifier()
forest_params = [{
    "n_estimators": [10, 25, 50, 75, 100],
    "criterion": ["entropy", "gini"],
    "class_weight": ["balanced", None],
    "bootstrap": [True, False],
    "max_depth": [6, 8, 10]
}]

forest_grid = GridSearchCV(rf_base, param_grid=forest_params, cv=5, scoring=recall_scorer, refit=False)
forest_grid.fit(X_train, y_train)

print("Best parameters:", forest_grid.best_params_)
print("Best CV weighted recall:", forest_grid.best_score_)

We train the final Random Forest model using the best parameters identified by `GridSearchCV`, then evaluate it on both the validation set and the held-out test set.

In [ ]:
rf_classifier = RandomForestClassifier(
    n_estimators=100,
    criterion='entropy',
    random_state=42,
    class_weight="balanced",
    bootstrap=False,
    max_depth=10
)
rf_classifier.fit(X_train, y_train)

# Validation set evaluation
y_val_pred_rf = rf_classifier.predict(X_val)
print("Random Forest — Validation Set")
print(classification_report(y_val, y_val_pred_rf, target_names=new_class))

In [ ]:
# Test set evaluation
y_test_pred_rf = rf_classifier.predict(X_test)
print("Random Forest — Test Set")
print(classification_report(y_test_combined, y_test_pred_rf, target_names=new_class))

### 3.1.2 Neural Network

A feedforward neural network is built using TensorFlow/Keras as an alternative to the tree-based approach. The architecture uses two hidden layers of 100 neurons each with ReLU activations and Dropout regularisation to reduce overfitting.

Key design decisions:
- **Class weights** are computed and passed during training to counteract the imbalanced class distribution
- **Early stopping** monitors validation loss with a patience of 10 epochs to prevent overfitting
- **Softmax output** over 4 classes with categorical cross-entropy loss

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.utils.class_weight import compute_class_weight

# One-hot encode labels
num_classes = len(np.unique(y_train))
y_train_ohe = to_categorical(y_train, num_classes)

# Compute class weights to handle imbalance
class_weights_arr = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
class_weights_dict = dict(enumerate(class_weights_arr))

# Build model
nn_model = Sequential([
    Dense(100, activation='relu', input_shape=(X_train.shape[1],)),
    Dropout(0.1),
    Dense(100, activation='relu'),
    Dropout(0.1),
    Dense(num_classes, activation='softmax')
])

nn_model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['Precision', 'Recall']
)

early_stopping = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

# Train
history = nn_model.fit(
    X_train, y_train_ohe,
    epochs=100,
    batch_size=32,
    validation_split=0.2,
    class_weight=class_weights_dict,
    callbacks=[early_stopping],
    verbose=1
)

In [ ]:
# Validation set evaluation
y_val_pred_nn = np.argmax(nn_model.predict(X_val), axis=1)
print("Neural Network — Validation Set")
print(classification_report(y_val, y_val_pred_nn, target_names=new_class))

# Test set evaluation
y_test_pred_nn = np.argmax(nn_model.predict(X_test), axis=1)
print("Neural Network — Test Set")
print(classification_report(y_test_combined, y_test_pred_nn, target_names=new_class))

## 3.2 Stage 2 Models: Binary Classification (Healthy vs Screening)

Stage 2 handles the binary problem of separating `healthy` from `screening stage cancer` — the most clinically sensitive distinction. This subset is highly imbalanced, with very few healthy samples relative to screening-stage ones.

Two algorithms are evaluated:
- **XGBoost** — a gradient boosted tree model tuned via `scale_pos_weight` to penalise misclassification of the minority class
- **Gaussian Naive Bayes** — a probabilistic baseline that assumes feature independence; fast and interpretable, particularly suited to continuous high-dimensional data

### 3.2.1 XGBoost

XGBoost's `scale_pos_weight` parameter controls how much extra weight is assigned to the minority class during training. We search over a range of large values (1000–4000) given the severe imbalance, using 5-fold stratified cross-validation optimised for F1 score.

In [ ]:
# Hyperparameter tuning for scale_pos_weight
model_xgb = XGBClassifier(eval_metric='logloss')

param_grid_xgb = {'scale_pos_weight': [1000, 2000, 3000, 4000]}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
grid_search_xgb = GridSearchCV(
    estimator=model_xgb,
    param_grid=param_grid_xgb,
    cv=cv,
    scoring="f1"
)
grid_search_xgb.fit(X_train2, y_2)

best_scale_pos_weight = grid_search_xgb.best_params_['scale_pos_weight']
print("Best scale_pos_weight:", best_scale_pos_weight)

# Train final XGBoost model with best hyperparameter
model_xgb_best = XGBClassifier(scale_pos_weight=best_scale_pos_weight, eval_metric='logloss')
model_xgb_best.fit(X_train2, y_2)

In [ ]:
y_pred_xgb = model_xgb_best.predict(X_test2)
print("XGBoost — Test Set")
print(classification_report(y_test2, y_pred_xgb, target_names=list(label_mapping.keys())))

### 3.2.2 Gaussian Naive Bayes

Gaussian Naive Bayes serves as an interpretable probabilistic baseline. It assumes each feature follows a Gaussian distribution within each class — a reasonable assumption for continuous biological features. We include a confusion matrix to visualise the model's error patterns, which is particularly informative for imbalanced binary classification.

In [ ]:
from sklearn.naive_bayes import GaussianNB

gnb_model = GaussianNB()
gnb_model.fit(X_train2, y_2)

y_pred_gnb = gnb_model.predict(X_test2)

print("Gaussian Naive Bayes — Test Set")
print(classification_report(y_test2, y_pred_gnb, target_names=list(label_mapping.keys())))

# Confusion matrix
conf_matrix = confusion_matrix(y_test2, y_pred_gnb)
disp = ConfusionMatrixDisplay(confusion_matrix=conf_matrix, display_labels=list(label_mapping.keys()))

fig, ax = plt.subplots(figsize=(5, 4))
disp.plot(ax=ax)
plt.title("Gaussian Naive Bayes — Confusion Matrix")
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

# 4. Summary

This notebook demonstrates a two-stage ML pipeline designed to handle a heavily imbalanced, high-dimensional cancer classification task.

| Stage | Task | Models Compared |
|-------|------|-----------------|
| Stage 1 | 4-class classification (healthy+screening / early / mid / late) | Random Forest, Neural Network |
| Stage 2 | Binary classification (healthy vs screening) | XGBoost, Gaussian Naive Bayes |

**Key design decisions:**
- Two-stage architecture to decompose a difficult 5-class imbalanced problem into manageable subproblems
- Class weighting and `scale_pos_weight` tuning to prevent models from ignoring minority classes
- Stratified cross-validation throughout to ensure evaluation is not biased by class imbalance
- Separate preprocessing for each stage (L2 normalisation for Stage 1; StandardScaler for Stage 2), with scalers fit only on training data to prevent leakage

**Technologies used:** Python, NumPy, pandas, scikit-learn, TensorFlow/Keras, XGBoost, Matplotlib